In [9]:
import pandas as pd
import json
import csv
import time
import os
import requests
from pathlib import Path

In [10]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "sk-XXXXXXXXX")
MODEL = "google/gemini-3.1-flash-lite-preview"
INPUT_JSONL = "alpaca_arabic_v5.jsonl"
OUTPUT_JSONL = "shami_dataset.jsonl"
INSTRUCTION_COL = "instruction"
OUTPUT_COL = "output"
INPUT_COL = "input"
DELAY_SECONDS = 1.0
MAX_ROWS = 3000

SYSTEM_PROMPT = """أنت مساعد متخصص في تحويل النصوص العربية الفصحى إلى اللهجة الشامية السورية.
قواعد مهمة:
- استخدم مفردات شامية أصيلة (هيك، شو، كيفك، بدي، عم، رح، هلق...)
- حافظ على المعنى الأصلي كاملاً
- لا تترجم الأسماء الأعلام
- الرد يكون النص الشامي فقط بدون أي شرح إضافي
- خلي الحكي مختصر و طبيعي
- حاول يكون الحكي كأنو نحنا بمسلسل باب الحارة"""


def translate_to_shami(text: str) -> str:
    if not text or not text.strip():
        return ""

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://github.com/your-repo",
    }

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"حول هالنص للهجة الشامية:\n{text}"}
        ],
        "max_tokens": 512,
        "temperature": 0.7,
    }

    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            json=payload,
            timeout=30
        )
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"].strip()

    except requests.exceptions.RequestException as e:
        print(f"  ERROR: {e}")
        return ""


def process_csv(input_path: str, output_path: str):
    input_file = Path(input_path)
    if not input_file.exists():
        print(f"File not Found: {input_path}")
        return
    
    with open(input_file, "r", encoding="utf-8") as f:
        rows = [line.rstrip("\n") for line in f]
        total_rows = len(rows)

    rows_to_process = min(total_rows, MAX_ROWS)
    print(f"File name: {input_path}")
    print(f"Total Lines: {total_rows} | Processing: {rows_to_process}")
    print(f"Model: {MODEL}")
    print("-" * 50)

    success_count = 0
    skip_count = 0

    with open(input_path, encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        rows = [line.rstrip("\n") for line in infile]

        for i, row in enumerate(rows):
            if i >= MAX_ROWS:
                break

            row = json.loads(row)

            instruction = row.get(INSTRUCTION_COL, "").strip()
            original_output = row.get(OUTPUT_COL, "").strip()
            input_text = row.get(INPUT_COL, "").strip()

            if not instruction or not original_output:
                skip_count += 1
                continue

            shami_instruction = translate_to_shami(instruction)
            time.sleep(DELAY_SECONDS)

            shami_output = translate_to_shami(original_output)
            time.sleep(DELAY_SECONDS)

            shami_input = ""
            if input_text:
                shami_input = translate_to_shami(input_text)
                time.sleep(DELAY_SECONDS)

            if not shami_instruction or not shami_output:
                print(f"Skipping line {i+1} due to empty translation.")
                skip_count += 1
                continue

            record = {
                "instruction": shami_instruction,
                "input": shami_input,
                "output": shami_output,
            }
            outfile.write(json.dumps(record, ensure_ascii=False) + "\n")
            success_count += 1

            if success_count % 50 == 0:
                print(f"Example: {i+1}: \n")
                print(f"  Instruction: {shami_instruction[:80]}...")
                print(f"  Output: {shami_output[:80]}...\n")

    print("-" * 50)
    print(f"Done!")
    print(f"   Success: {success_count}")
    print(f"   Skipped: {skip_count}")
    print(f"   Saved to: {output_path}")


if __name__ == "__main__":
    process_csv(INPUT_JSONL, OUTPUT_JSONL)

File name: alpaca_arabic_v5.jsonl
Total Lines: 52002 | Processing: 3000
Model: google/gemini-3.1-flash-lite-preview
--------------------------------------------------
  ERROR: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions
  ERROR: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions
Skipping line 1 due to empty translation.
  ERROR: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions
  ERROR: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions
Skipping line 2 due to empty translation.
  ERROR: 401 Client Error: Unauthorized for url: https://openrouter.ai/api/v1/chat/completions


KeyboardInterrupt: 